## A. Load dataset & preprocessing sama seperti Part 2

In [6]:
import numpy as np
import torch
import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

iris = load_iris(as_frame=True)
X, y = iris.data.copy(), iris.target.copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Jumlah sampel:', len(X), '| Train:', len(X_train), '| Test:', len(X_test))
print('Nilai kosong:', int(X.isna().sum().sum()))
print('Versi scikit-learn:', sklearn.__version__, '| PyTorch:', torch.__version__)


Jumlah sampel: 150 | Train: 120 | Test: 30
Nilai kosong: 0
Versi scikit-learn: 1.6.1 | PyTorch: 2.14.1+cpu


## B. Melatih & tuning KNN (5-fold CV, F1-macro) - konsisten Part 2

In [7]:
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier()),
])
param_grid = {
    'model__n_neighbors': [3, 5, 7, 9],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(knn_pipeline, param_grid, scoring='f1_macro', cv=cv, n_jobs=1)
search.fit(X_train, y_train)
best = search.best_estimator_
print('Hyperparameter terbaik:', search.best_params_)
print(f'5-fold CV Macro F1 : {search.best_score_:.4f}')

pred_sklearn = best.predict(X_test)
print(f'Akurasi test         : {accuracy_score(y_test, pred_sklearn):.4f}')
print(f'F1-macro test        : {f1_score(y_test, pred_sklearn, average="macro"):.4f}')
print('Confusion Matrix:')
print(confusion_matrix(y_test, pred_sklearn))


Hyperparameter terbaik: {'model__n_neighbors': 3, 'model__p': 2, 'model__weights': 'uniform'}
5-fold CV Macro F1 : 0.9665
Akurasi test         : 0.9333
F1-macro test        : 0.9327
Confusion Matrix:
[[10  0  0]
 [ 0 10  0]
 [ 0  2  8]]


## C. Ekspor checkpoint model `.pth` dengan PyTorch

In [8]:
from pathlib import Path

scaler = best.named_steps['scaler']
knn = best.named_steps['model']

checkpoint = {
    'train_features_scaled': torch.tensor(
        scaler.transform(X_train), dtype=torch.float64
    ),
    'train_labels': torch.tensor(y_train.to_numpy(), dtype=torch.int64),
    'scaler_mean': torch.tensor(scaler.mean_, dtype=torch.float64),
    'scaler_scale': torch.tensor(scaler.scale_, dtype=torch.float64),
    'k': int(knn.n_neighbors),
    'p': int(knn.p),
    'class_names': [str(name) for name in iris.target_names],
    'feature_names': [str(col) for col in iris.feature_names],
}
model_path = Path('iris_knn_tuned.pth')
torch.save(checkpoint, model_path)
print('File model dibuat:', model_path.resolve())
print('Ukuran file:', model_path.stat().st_size, 'bytes')


File model dibuat: /content/iris_knn_tuned.pth
Ukuran file: 7459 bytes


In [9]:
loaded = torch.load('iris_knn_tuned.pth', map_location='cpu', weights_only=True)

def prediksi_torch(inputs, state):
    inputs_t = torch.tensor(inputs, dtype=torch.float64)
    standardized = (inputs_t - state['scaler_mean']) / state['scaler_scale']
    distances = torch.linalg.vector_norm(
        state['train_features_scaled'] - standardized, ord=state['p'], dim=1
    )
    nearest = torch.argsort(distances, stable=True)[:state['k']]
    votes = torch.bincount(state['train_labels'][nearest], minlength=3)
    return int(torch.argmax(votes))

pred_pth = np.array([prediksi_torch(row, loaded) for row in X_test.to_numpy()])
assert np.array_equal(pred_pth, pred_sklearn), 'Prediksi checkpoint .pth berbeda dari sklearn!'
print('VERIFIKASI LULUS: semua 30 prediksi .pth identik dengan KNN Scikit-learn.')
print('Contoh prediksi [5.1,3.5,1.4,0.2]:', loaded['class_names'][
    prediksi_torch([5.1, 3.5, 1.4, 0.2], loaded)
])


VERIFIKASI LULUS: semua 30 prediksi .pth identik dengan KNN Scikit-learn.
Contoh prediksi [5.1,3.5,1.4,0.2]: setosa


In [10]:

try:
    from google.colab import files
    files.download('iris_knn_tuned.pth')
except ModuleNotFoundError:
    print('Bukan Google Colab: file iris_knn_tuned.pth sudah tersimpan di folder kerja.')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>